In [11]:
!pip install dagshub mlflow optuna -q

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

dagshub.init (
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("Decision Trees")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


In [13]:
with mlflow.start_run(run_name="DecisionTrees_Cleaning"):
    import gc
    import numpy as np
    import pandas as pd

    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD = 0.80
    ID_MISSING_THRESHOLD = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    # load
    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    # fix id-01 vs id_01
    test_id.columns = test_id.columns.str.replace("-", "_", regex=False)

    # merge
    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id, on="TransactionID", how="left")

    del train_txn, train_id, test_txn, test_id
    gc.collect()

    # split target
    y_train = train["isFraud"].astype(np.int8)
    X_train = train.drop(columns=["isFraud", "TransactionID"])
    X_test  = test.drop(columns=["TransactionID"])

    del train, test
    gc.collect()

    # drop high-missing
    id_like_cols = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train.columns if c not in id_like_cols]
    missing_ratio = X_train.isnull().mean()

    drop_txn = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id  = [c for c in id_like_cols if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing = drop_txn + drop_id

    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    # drop near-constant
    near_constant_cols = []
    for col in X_train.columns:
        top_freq = X_train[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train = X_train.drop(columns=near_constant_cols)
    X_test  = X_test.drop(columns=[c for c in near_constant_cols if c in X_test.columns])

    # align test columns
    for col in X_train.columns:
        if col not in X_test.columns:
            X_test[col] = np.nan
    X_test = X_test[X_train.columns]

    # log
    mlflow.log_param("txn_missing_threshold", TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold", ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)

    mlflow.log_metric("train_rows", int(X_train.shape[0]))
    mlflow.log_metric("test_rows", int(X_test.shape[0]))
    mlflow.log_metric("final_features", int(X_train.shape[1]))
    mlflow.log_metric("fraud_rate", float(y_train.mean()))
    mlflow.log_metric("dropped_missing_cols", int(len(drop_missing)))
    mlflow.log_metric("dropped_near_constant_cols", int(len(near_constant_cols)))

    print(f"X_train_clean: {X_train.shape}")
    print(f"X_test_clean:  {X_test.shape}")

    # keep for next cells
    X_train_clean = X_train
    X_test_clean = X_test
    y_train_clean = y_train

X_train_clean: (590540, 353)
X_test_clean:  (506691, 353)
🏃 View run DecisionTrees_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/7a016fdbb84c4a419d4514ea0aac2ad1
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/2


In [14]:
with mlflow.start_run(run_name="DecisionTree_FeatureEngineering"):
    from sklearn.impute import SimpleImputer

    X_train = X_train_clean.copy()
    X_test = X_test_clean.copy()
    y_train = y_train_clean.copy()

    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"] = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"] = np.sin(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_cos"] = np.cos(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test = X_test.drop(columns=["TransactionDT"], errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    num_imp = SimpleImputer(strategy="median")
    X_train[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test[num_cols] = num_imp.transform(X_test[num_cols])

    cat_maps = {}
    for c in cat_cols:
        uniq = pd.Series(X_train[c].astype(str).unique())
        mapping = {v: i for i, v in enumerate(uniq)}
        cat_maps[c] = mapping
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c] = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_metric("features_after_fe", int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded", int(len(cat_cols)))

    print("X_train_fe:", X_train.shape)
    print("X_test_fe: ", X_test.shape)

    X_train_fe_dt = X_train
    X_test_fe_dt = X_test
    y_train_fe_dt = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run DecisionTree_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/e46d63959fcf4d1586779ba498d8208c
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/2


In [15]:
print(X_train_fe_dt.shape, X_test_fe_dt.shape)
assert X_train_fe_dt.shape[1] == X_test_fe_dt.shape[1]
print("object cols left:", X_train_fe_dt.select_dtypes(include=["object"]).shape[1])

(590540, 355) (506691, 355)
object cols left: 0


In [16]:
with mlflow.start_run(run_name="DecisionTree_FeatureSelection"):
    X_train = X_train_fe_dt.copy()
    X_test = X_test_fe_dt.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test = X_test.drop(columns=const_cols, errors="ignore")

    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    sample_n = min(120_000, len(X_train))
    idx = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()

    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > 0.98).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test = X_test.drop(columns=drop_corr, errors="ignore")

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_metric("dropped_const", len(const_cols))
    mlflow.log_metric("dropped_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print("X_train_fs:", X_train.shape)

    X_train_final_dt = X_train
    X_test_final_dt = X_test

X_train_fs: (590540, 306)
🏃 View run DecisionTree_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/ffba770061b84c9e84636c4b8bfd9757
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/2


In [17]:
print(X_train_final_dt.shape, X_test_final_dt.shape)
assert X_train_final_dt.shape[1] == X_test_final_dt.shape[1]

(590540, 306) (506691, 306)
